# Notebook: Wikidata Relation Data Preparation
This notebook generates a relational tensor capturing complex, real-world connections between companies, as defined in the Wikidata knowledge graph. Unlike basic sector or industry relationships, these connections may involve multi-step paths (e.g., "Company A was founded by Person X, who also serves on the board of Company B").

**Output files**:

* A 3D NumPy array with shape `(num_tickers, num_tickers, num_relation_paths + 1)`. Each slice along the third dimension represents a specific type of relational path. A value of 1 at position `(i, j, k)` indicates the presence of that relational path between stock *i* and stock *j*.


## Imports and paths

In [4]:
import os
import json
import numpy as np

In [5]:
market_name = 'NASDAQ'
relation_path = os.path.join('..', '..', 'data', 'relation', 'relation', 'wikidata')
tic_wiki_path = os.path.join('..', '..', 'data')
tic_wiki_file = os.path.join(tic_wiki_path, f'{market_name}_wiki.csv')
sel_path_file = os.path.join(relation_path, 'selected_wiki_connections.csv')
connection_file = os.path.join(relation_path, f'{market_name}_connections.json')

## Ticker-to-WikiID Mappings

The ticker-to-WikiID mapping file is first loaded. Since not all tickers correspond to a Wikidata entry, a dictionary `wikiid_ticind_dic` is constructed to map each available WikiID to the numerical index of its associated ticker. This enables translation of relationships from the knowledge graph into the stock matrix. Out of 1026 tickers, 512 possess valid WikiIDs.


In [6]:
# Load Tickers
tickers = np.genfromtxt(tic_wiki_file, dtype=str, delimiter=',', skip_header=False)
print('#tickers selected:', tickers.shape)

#tickers selected: (1026, 2)


In [4]:
tickers

array([['AABA', 'Q30321616'],
       ['AAON', 'Q286214'],
       ['AAPL', 'Q312'],
       ...,
       ['ZION', 'Q8072579'],
       ['ZIV', 'unknown'],
       ['ZUMZ', 'Q8075252']], dtype='<U9')

In [8]:
# Map WikiIDs to Index
wikiid_ticind_dic = {}
for ind, tw in enumerate(tickers):
    if tw[-1] != 'unknown':
        wikiid_ticind_dic[tw[-1]] = ind
print('#tickers aligned:', len(wikiid_ticind_dic))

#tickers aligned: 512


In [ ]:
wikiid_ticind_dic

## Load and filter relational paths

Relational paths are processed in three steps:

1. **Load Selected Paths**: The file `selected_wiki_connections.csv` is loaded, containing a curated list of 57 meaningful path types (e.g., `P127_P127` indicating shared ownership).

2. **Load Connection Graph**: The file `connections.json` is loaded, representing the full graph of all discovered relational paths between entities.

3. **Identify Occurring Paths**: The full connection graph is iterated to determine which of the 57 selected path types actually occur among the relevant tickers. This filtering reduces the set to 42 occurring paths, increasing the density and efficiency of the final tensor.


In [10]:
# Load Selected Wiki Paths
sel_paths = np.genfromtxt(sel_path_file, dtype=str, delimiter=' ', skip_header=False)
print('#paths selected:', len(sel_paths))
sel_paths = set(sel_paths[:, 0])

#paths selected: 57


In [ ]:
sel_paths

In [14]:
# Load Connection Graph: loads the full graph of entity connections from the Wikidata-derived JSON
with open(connection_file, 'r') as f:
    connections = json.load(f)
print('#connection items:', len(connections))

#connection items: 511


In [ ]:
connections

In [16]:
# Filter Paths Present in Connection Graph: iterates through all connections 
# and keeps only those that match the selected paths
occur_paths = set()
for sou_item, conns in connections.items():
    for tar_item, paths in conns.items():
        for p in paths:
            path_key = '_'.join(p)
            if path_key in sel_paths:
                occur_paths.add(path_key)

print('#occurred selected paths:', len(occur_paths))

#occurred selected paths: 42


In [ ]:
occur_paths

## Construct the Relational Tensor
Now, we build the final multi-hot encoded tensor wiki_relation_embedding.
1. Index Valid Paths: We create a dictionary valid_path_index to map each of the 42 occurring path types to a unique integer index (from 0 to 41).
2. Initialize Tensor: We create a zero-filled tensor of shape (1026, 1026, 43). The last dimension (42+1) corresponds to our 42 path types plus one channel for the self-relation flag.
3. Populate Tensor: We iterate through the connection graph again. For each connection between a source sou_item and target tar_item via a valid path path_key, we find their corresponding ticker indices i and j. We then set the value at tensor[i][j][k] to 1, where k is the index of path_key.
4. Set Self-Relations: Finally, we set the last element of the diagonal, tensor[i][i][-1], to 1 for all tickers to encode a self-relationship.

In [ ]:
# Index Valid Paths: builds an index mapping from path name to index
valid_path_index = {path: i for i, path in enumerate(occur_paths)}
for path, i in valid_path_index.items():
    print(path, i)

In [19]:
# Creates a zero tensor to store relations: [num_tickers, num_tickers, num_paths + 1]
wiki_relation_embedding = np.zeros(
    (tickers.shape[0], tickers.shape[0], len(valid_path_index) + 1),
    dtype=int
)

In [20]:
# Populate Relation Embedding
conn_count = 0
for sou_item, conns in connections.items():
    for tar_item, paths in conns.items():
        for p in paths:
            path_key = '_'.join(p)
            if path_key in valid_path_index:
                if sou_item in wikiid_ticind_dic and tar_item in wikiid_ticind_dic:
                    i = wikiid_ticind_dic[sou_item]
                    j = wikiid_ticind_dic[tar_item]
                    k = valid_path_index[path_key]
                    wiki_relation_embedding[i][j][k] = 1
                    conn_count += 1

print('connections count:', conn_count,
      'ratio:', conn_count / float(tickers.shape[0] * tickers.shape[0]))

connections count: 2171 ratio: 0.0020623629682827386


In [21]:
# Set Self-Relations: ensure each ticker has a self-relation encoded in the final channel
for i in range(tickers.shape[0]):
    wiki_relation_embedding[i][i][-1] = 1

print('Final relation tensor shape:', wiki_relation_embedding.shape)

Final relation tensor shape: (1026, 1026, 43)


In [ ]:
# Saves the relation tensor to disk as a NumPy binary file
# output_file = f'{market_name}_wiki_relation.npy'
# np.save(output_file, wiki_relation_embedding)
# print(f'Saved: {output_file}')